# CASEN 2024 Income Prediction Challenge
## Data Mining - Doctorado UDP 2026
### Dr. Bastián González-Bustamante, Profesor Asociado

Evaluación de los envíos (uso del profesor). Para cada archivo de `submissions/`:

1. Valida el nombre (`submission_<Nombre>_<Modelo>_<N>.csv`, con un `<Nombre>` de la lista `PARTICIPANTS`) y el formato (columnas exactamente `id,predicted_label`; todos los `id` del test una sola vez; etiquetas dentro de las cinco clases). Los archivos inválidos se mueven a `submissions_invalid/`.
2. Fecha el envío con la fecha del commit que agregó el archivo a `submissions/` (`git log`; si volvió desde standby, la del día en que se reincorporó) y, si no está en git, con la fecha de modificación del archivo (hora de Chile). Los envíos fuera de plazo se consideran inválidos.
3. Aplica la regla de **un envío por participante por día**: el primero del día se evalúa y los demás se mueven a `submissions_standby/`.
4. Calcula accuracy y precision, recall y F1 macro contra `data/y_test.csv` (privado, no está en GitHub), escribe `leaderboard.csv` (historial completo) y `leaderboard_best.csv` (mejor envío por participante, **sin notas**) e imprime las tablas en Markdown para el README.
5. Calcula la nota de cada participante y la guarda **solo en local**, en `notas_privadas.csv` (ignorado por git). Las notas nunca se publican: ni en los CSV públicos, ni en el README, ni en este notebook si se sube a GitHub.

In [1]:
## Parámetros del challenge
LAUNCH_DATE = "2026-11-10"        ## Lanzamiento: martes 10 de noviembre de 2026 (clase recuperativa de la Sesión 11)
DEADLINE = "2026-11-30 23:59"     ## Cierre: lunes 30 de noviembre de 2026, 23:59 hora de Chile (UTC−3)
K = 11.81                          ## Nota = 1 + (mejor F1 − 0.001) · K, acotada a 7.0
TIMEZONE = "America/Santiago"
EXEMPT = {"Baseline"}              ## Participantes sin control de plazo ni de un envío por día
## Nombres aceptados: el primer nombre (o el alias acordado) de cada estudiante, sin tildes, espacios
## ni guiones bajos, como figura en la lista de participantes del README. Un archivo con otro nombre (p. ej., "Dayana2") no se
## evalúa y se mueve a submissions_invalid/. Completar antes del lanzamiento.
PARTICIPANTS = {"Carolina", "Karla", "MariaCamila", "Marian", "Matias", "Tamara"}  ## 21 oct 2026
MOVE_FILES = True                  ## Mover inválidos a submissions_invalid/ y extras a submissions_standby/
DATE_OVERRIDES = {"submission_MariaCamila_RF100_1.csv": "2026-11-10T17:59:43"}  ## Correcciones manuales justificadas: {"submission_X_Y_N.csv": "AAAA-MM-DD"}

SUBMISSIONS_DIR = "submissions"
INVALID_DIR = "submissions_invalid"
STANDBY_DIR = "submissions_standby"

In [2]:
## Dependencias
import os
import re
import shutil
import subprocess
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    from zoneinfo import ZoneInfo
    TZ = ZoneInfo(TIMEZONE)
except Exception:  ## sin base de zonas horarias (e.g., Windows sin tzdata): UTC-3, horario de verano de Chile
    TZ = timezone(timedelta(hours=-3))

launch = datetime.fromisoformat(LAUNCH_DATE).replace(tzinfo=TZ)
deadline = datetime.fromisoformat(DEADLINE).replace(tzinfo=TZ) + timedelta(seconds=59)

## Datos de entrenamiento y verdad de terreno del test set (no disponible en GitHub)
target = "ypchautcor"
labels_all = ['Low', 'Medium-Low', 'Medium', 'Medium-High', 'High']
if not os.path.exists("data/y_test.csv"):
    raise FileNotFoundError("Falta data/y_test.csv (privado). Cópienlo localmente; nunca lo suban al repositorio.")
y_train = pd.read_csv("data/y_train.csv")
y_test = pd.read_csv("data/y_test.csv")

## Discretizar SOLO con los quintiles de train (idéntico a baseline_model.ipynb)
_, bins = pd.qcut(y_train[target], q=5, retbins=True, duplicates='drop')
bins[0], bins[-1] = -np.inf, np.inf
labels = labels_all[:len(bins) - 1]
y_test[target] = pd.cut(y_test[target], bins=bins, labels=labels, include_lowest=True).astype(str)
truth = y_test.set_index("id")[target]
print("Puntos de corte (CLP/mes):", [round(b) for b in bins[1:-1]], "| test:", truth.shape[0], "filas")

Puntos de corte (CLP/mes): [132852, 250000, 389118, 655000] | test: 11799 filas


In [3]:
NAME_RE = re.compile(r"^submission_([A-Za-z0-9ÁÉÍÓÚÑáéíóúñ-]+)_([A-Za-z0-9.+-]+)_([0-9]+)\.csv$")


def parse_name(file_name):
    m = NAME_RE.match(file_name)
    return (m.group(1), m.group(2), int(m.group(3))) if m else None


def submission_time(path):
    """Fecha del commit que agregó el archivo a submissions/ (fecha de autor, hora de Chile).

    Normalmente hay un solo commit que agrega el archivo. Si un envío volvió desde
    submissions_standby/, se usa el agregado más reciente (el día en que se reincorpora).
    Si el archivo no está en git, se usa su fecha de modificación.
    """
    try:
        out = subprocess.run(
            ["git", "log", "--diff-filter=A", "--format=%aI", "--", path],
            capture_output=True, text=True, check=True,
        ).stdout.split()
        if out:
            return datetime.fromisoformat(out[0]).astimezone(TZ), "git"  ## git log: más reciente primero
    except (OSError, subprocess.CalledProcessError):
        pass
    return datetime.fromtimestamp(os.path.getmtime(path), tz=TZ), "mtime"


def validate(sub):
    """Lista de errores de formato (vacía si el envío es válido)."""
    errors = []
    if list(sub.columns) != ["id", "predicted_label"]:
        return [f"columnas {list(sub.columns)} en vez de ['id', 'predicted_label']"]
    ids = pd.to_numeric(sub["id"], errors="coerce")
    if ids.isna().any() or (ids % 1 != 0).any():
        errors.append("hay id no enteros o vacíos")
    else:
        ids = ids.astype("int64")
        if ids.duplicated().any():
            errors.append(f"{int(ids.duplicated().sum())} id duplicados")
        missing = set(truth.index) - set(ids)
        extra = set(ids) - set(truth.index)
        if missing:
            errors.append(f"faltan {len(missing)} id del test set")
        if extra:
            errors.append(f"{len(extra)} id que no existen en el test set")
    bad = set(sub["predicted_label"].dropna().astype(str)) - set(labels_all)
    if sub["predicted_label"].isna().any():
        errors.append(f"{int(sub['predicted_label'].isna().sum())} etiquetas vacías")
    if bad:
        errors.append(f"etiquetas fuera de las cinco clases: {sorted(bad)[:5]}")
    return errors


def move(file_name, dest_dir):
    if MOVE_FILES:
        os.makedirs(dest_dir, exist_ok=True)
        shutil.move(os.path.join(SUBMISSIONS_DIR, file_name), os.path.join(dest_dir, file_name))

In [4]:
## 1. Validar nombre, formato y plazo
records, invalid = [], []
if not isinstance(DATE_OVERRIDES, dict):
    raise TypeError('DATE_OVERRIDES debe ser un diccionario {"submission_X_Y_N.csv": "AAAA-MM-DD"} (dos puntos, no coma)')
for f, d in DATE_OVERRIDES.items():
    datetime.fromisoformat(d)  ## falla si la fecha no es ISO
    if not os.path.exists(os.path.join(SUBMISSIONS_DIR, f)):
        print(f"AVISO  DATE_OVERRIDES: {f} no está en {SUBMISSIONS_DIR}/ (¿falta .csv o ya se movió?)")
pending = [f for f in os.listdir(SUBMISSIONS_DIR) if f.endswith(".csv") and (parse_name(f) or ("",))[0] not in EXEMPT]
if pending and not PARTICIPANTS:
    raise ValueError("PARTICIPANTS está vacío: complete la lista de nombres aceptados antes de evaluar envíos de estudiantes.")
for file_name in sorted(os.listdir(SUBMISSIONS_DIR)):
    if not file_name.endswith(".csv"):
        continue
    path = os.path.join(SUBMISSIONS_DIR, file_name)
    parsed = parse_name(file_name)
    if parsed is None:
        invalid.append((file_name, "nombre no sigue submission_<Nombre>_<Modelo>_<N>.csv"))
        continue
    participant, model_name, number = parsed
    if participant not in PARTICIPANTS | EXEMPT:
        invalid.append((file_name, f"participante no registrado: {participant} (ver PARTICIPANTS)"))
        continue
    try:
        sub = pd.read_csv(path, dtype={"predicted_label": str})
    except Exception as e:
        invalid.append((file_name, f"no se pudo leer el CSV ({e})"))
        continue
    errors = validate(sub)
    if errors:
        invalid.append((file_name, "; ".join(errors)))
        continue
    when, source = submission_time(path)
    if file_name in DATE_OVERRIDES:
        when, source = datetime.fromisoformat(DATE_OVERRIDES[file_name]).replace(tzinfo=TZ), "override"
    if participant not in EXEMPT and not (launch <= when <= deadline):
        invalid.append((file_name, f"fuera de plazo ({when:%Y-%m-%d %H:%M})"))
        continue
    records.append({"file": file_name, "Participant": participant, "Model": model_name, "Submission": number,
                    "timestamp": when, "Date": when.strftime("%Y-%m-%d"), "date_source": source, "sub": sub})

for file_name, reason in invalid:
    print(f"INVÁLIDO  {file_name}: {reason}")
    move(file_name, INVALID_DIR)
print(f"{len(records)} envíos con formato válido, {len(invalid)} inválidos")

16 envíos con formato válido, 0 inválidos


In [5]:
## 2. Máximo un envío por participante por día: se evalúa el primero; los demás quedan en standby
standby = []
if records:
    order = sorted(records, key=lambda r: (r["Participant"], r["Date"], r["timestamp"], r["Submission"]))
    seen = set()
    kept = []
    for r in order:
        key = (r["Participant"], r["Date"])
        if key in seen and r["Participant"] not in EXEMPT:
            standby.append(r)
        else:
            seen.add(key)
            kept.append(r)
    records = kept
for r in standby:
    print(f"STANDBY   {r['file']}: segundo envío de {r['Participant']} el {r['Date']}")
    move(r["file"], STANDBY_DIR)
print(f"{len(records)} envíos evaluados, {len(standby)} en standby")

16 envíos evaluados, 0 en standby


In [6]:
## 3. Métricas contra el test set privado
def grade(f1_value, k=K):
    return float(np.clip(1.0 + (f1_value - 0.001) * k, 1.0, 7.0))


results = []
for r in records:
    pred = r["sub"].assign(id=lambda d: d["id"].astype("int64")).set_index("id")["predicted_label"].reindex(truth.index)
    results.append({
        "Participant": r["Participant"],
        "Date": r["Date"],
        "Submission": r["Submission"],
        "Model": r["Model"],
        "Accuracy": accuracy_score(truth, pred),
        "Precision": precision_score(truth, pred, average="macro", labels=labels, zero_division=0),
        "Recall": recall_score(truth, pred, average="macro", labels=labels, zero_division=0),
        "F1-Score": f1_score(truth, pred, average="macro", labels=labels, zero_division=0),
    })

cols = ["Participant", "Date", "Submission", "Model", "Accuracy", "Precision", "Recall", "F1-Score"]
leaderboard = pd.DataFrame(results, columns=cols).sort_values(["F1-Score", "Date"], ascending=[False, True])
best = leaderboard.drop_duplicates("Participant").reset_index(drop=True)

## Actualizar los leaderboards públicos (sin notas)
leaderboard.to_csv("leaderboard.csv", index=False)
best.to_csv("leaderboard_best.csv", index=False)

## Notas: solo en un archivo local, ignorado por git (**/notas_privadas.csv); nunca se publican
notas = best.assign(Nota=best["F1-Score"].map(grade).round(2))[["Participant", "F1-Score", "Nota"]]
notas.to_csv("notas_privadas.csv", index=False)
print(leaderboard.to_string(index=False))

Participant       Date  Submission      Model  Accuracy  Precision   Recall  F1-Score
MariaCamila 2026-11-12           3   LightGBM  0.514196   0.524589 0.513276  0.514882
      Karla 2026-11-12           2    XGBoost  0.515044   0.515811 0.515173  0.513950
     Tamara 2026-11-13           3 XGBoost500  0.515891   0.509536 0.513834  0.509302
   Carolina 2026-11-13           3      RFMLP  0.514705   0.506605 0.513501  0.508361
MariaCamila 2026-11-11           2   LightGBM  0.513349   0.507321 0.511489  0.507550
     Matias 2026-11-11           2   LightGBM  0.512755   0.505287 0.511638  0.507227
     Tamara 2026-11-12           2      RF200  0.512077   0.505821 0.511486  0.507158
   Carolina 2026-11-10           1    XGBoost  0.513433   0.506185 0.511788  0.507137
   Carolina 2026-11-11           2      HGBx5  0.513094   0.505658 0.511389  0.506742
     Marian 2026-11-12           2     RF1000  0.508772   0.502381 0.507778  0.503803
     Matias 2026-11-10           1      RF500  0.50877

In [7]:
## 4. Tablas en Markdown para pegar en el README
def md_table(df):
    head = cols
    lines = [" | ".join(head), " | ".join(["---"] + [":-:"] * (len(head) - 1))]
    top = df.loc[df["Participant"] != "Baseline", "Participant"].head(1).tolist()
    trophy_done = False
    for _, r in df.iterrows():
        name = r["Participant"]
        if top and name == top[0] and not trophy_done:
            name, trophy_done = name + " :trophy:", True
        row = [name, r["Date"], str(r["Submission"]), r["Model"]] + [f"{r[c]:.3f}" for c in cols[4:]]
        lines.append(" | ".join(row))
    return "\n".join(lines)


print("### Mejor envío por participante\n")
print(md_table(best))
print("\n### Historial completo\n")
print(md_table(leaderboard))

### Mejor envío por participante

Participant | Date | Submission | Model | Accuracy | Precision | Recall | F1-Score
--- | :-: | :-: | :-: | :-: | :-: | :-: | :-:
MariaCamila :trophy: | 2026-11-12 | 3 | LightGBM | 0.514 | 0.525 | 0.513 | 0.515
Karla | 2026-11-12 | 2 | XGBoost | 0.515 | 0.516 | 0.515 | 0.514
Tamara | 2026-11-13 | 3 | XGBoost500 | 0.516 | 0.510 | 0.514 | 0.509
Carolina | 2026-11-13 | 3 | RFMLP | 0.515 | 0.507 | 0.514 | 0.508
Matias | 2026-11-11 | 2 | LightGBM | 0.513 | 0.505 | 0.512 | 0.507
Marian | 2026-11-12 | 2 | RF1000 | 0.509 | 0.502 | 0.508 | 0.504
Baseline | 2026-11-10 | 1 | RF100 | 0.257 | 0.196 | 0.253 | 0.210

### Historial completo

Participant | Date | Submission | Model | Accuracy | Precision | Recall | F1-Score
--- | :-: | :-: | :-: | :-: | :-: | :-: | :-:
MariaCamila :trophy: | 2026-11-12 | 3 | LightGBM | 0.514 | 0.525 | 0.513 | 0.515
Karla | 2026-11-12 | 2 | XGBoost | 0.515 | 0.516 | 0.515 | 0.514
Tamara | 2026-11-13 | 3 | XGBoost500 | 0.516 | 0.510 | 0.5